# 🚀 DT-RL: End-to-End Automated Training Pipeline

**Sync → Train → Track → Push**

This notebook is designed to run the entire DT-RL training lifecycle autonomously:
1. Pulls the latest code from your GitHub repository.
2. Installs all required dependencies.
3. Authenticates with Weights & Biases (WandB) for tracking.
4. Executes the training script.
5. Commits the new checkpoints and generated visualization plots.
6. Pushes the updates back to your GitHub repository.

> **Prerequisites**:
> You must set up the following "Secrets" in Google Colab (the key icon on the left sidebar):
> - `GITHUB_TOKEN`: A GitHub Personal Access Token with repo scope.
> - `WANDB_API_KEY`: Your Weights & Biases API Key.

## 1. ⚙️ Configuration & GitHub Sync

In [ ]:
import os
import subprocess
from google.colab import userdata

# ==========================================
# CONFIGURATION
# ==========================================
GITHUB_USER = "VvS-2403"                   # Your GitHub username
GITHUB_REPO = "DT-RL"                      # The repository name
GITHUB_EMAIL = "your-email@example.com"    # Email for git commits
GITHUB_NAME = "Automated Colab Runner"     # Name for git commits
BRANCH = "main"                            # Branch to pull from / push to

# Training hyperparameters (passed to train_wandb.py)
TRAIN_ARGS = [
    "--epochs", "50",
    "--batch-size", "16",
    "--num-assets", "20",
    "--lookback", "60",
    "--horizon", "5",
    "--output-mode", "portfolio_weights"
]
# ==========================================

# 1. Fetch GitHub Token from Colab Secrets
try:
    GIT_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ GitHub Token found in Colab Secrets.")
except Exception as e:
    raise ValueError("❌ GITHUB_TOKEN secret not found! Please add it in Colab Secrets.")

# Construct the repository URL with the access token
REPO_URL = f"https://{GIT_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# 2. Clone or Update the Repository
if not os.path.exists(GITHUB_REPO):
    print(f"📥 Cloning repository: {GITHUB_REPO}...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
else:
    print(f"🔄 Repository {GITHUB_REPO} exists. Pulling latest changes...")
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=GITHUB_REPO, check=True)

# Change working directory to the repo
os.chdir(GITHUB_REPO)
print(f"📁 Current working directory: {os.getcwd()}")

## 2. 📦 Install Dependencies

In [ ]:
print("📦 Installing dependencies from requirements.txt...")
# Install requirements
!pip install -q -r requirements.txt
print("✅ Dependencies installed.")

## 3. 🔑 Weights & Biases Authentication

In [ ]:
import wandb

# Fetch WandB API Key from Colab Secrets
try:
    WANDB_KEY = userdata.get('WANDB_API_KEY')
    print("✅ WandB API Key found in Colab Secrets.")
except Exception as e:
    raise ValueError("❌ WANDB_API_KEY secret not found! Please add it in Colab Secrets.")

# Login to WandB
os.environ["WANDB_API_KEY"] = WANDB_KEY
wandb.login(key=WANDB_KEY)
print("✅ Successfully authenticated with Weights & Biases.")

## 4. 🏋️ Run Training Pipeline

This executes `train_wandb.py` using the arguments defined in the configuration.

In [ ]:
print(f"🚀 Starting training run...")
train_command = ["python", "train_wandb.py"] + TRAIN_ARGS

# Execute the training script
!{" ".join(train_command)}

print("✅ Training completed.")

## 5. 📤 Push Results to GitHub

Commits newly generated checkpoints (`checkpoints/`) and visualizations (`results/`) back to the repository.

In [ ]:
print("⚙️ Configuring git user...")
subprocess.run(["git", "config", "--global", "user.email", GITHUB_EMAIL], check=True)
subprocess.run(["git", "config", "--global", "user.name", GITHUB_NAME], check=True)

print("📝 Adding new files to git tracking...")
# Add specific directories that contain outputs
subprocess.run(["git", "add", "checkpoints/"], check=True)
subprocess.run(["git", "add", "results/"], check=True)

# Check if there are changes to commit
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)

if status.stdout.strip():
    print("📦 Committing changes...")
    import datetime
    commit_msg = f"Automated training run results - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    subprocess.run(["git", "commit", "-m", commit_msg], check=True)
    
    print("📤 Pushing to GitHub...")
    # Push to the branch (requires the URL with token)
    subprocess.run(["git", "push", REPO_URL, BRANCH], check=True)
    print("✅ Successfully pushed updates to GitHub.")
else:
    print("ℹ️ No new changes or results to commit.")

## 🎉 Workflow Complete
Your model has been trained, tracked in WandB, and the artifacts are safely stored in your GitHub repository!